# Exploratory Data Analysis — Telco Customer Churn
This notebook explores the IBM Telco Customer Churn dataset.
With an approximately ~26% churn rate, understanding the key drivers is worth
millions in targeted retention spend. Here I analyse feature distributions,
class imbalance, and correlations before modelling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## Dataset shape and churn rate
First we check how many customers we have and what proportion churned.
A heavily imbalanced dataset would require resampling techniques.

In [ ]:
print(f"Dataset: {df.shape[0]:,} customers, {df.shape[1]} features")
print(f"Churn rate: {df['Churn'].value_counts(normalize=True)['Yes']:.1%}")

## Class imbalance
Visualising the split between churned and retained customers.
A ~26% churn rate means the dataset is moderately imbalanced, so
we will use stratified splits during modelling to preserve this ratio.

In [ ]:
df['Churn'].value_counts().plot(kind='bar', color=['#185FA5','#E24B4A'])
plt.title('Customer churn distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Numeric feature distributions
Checking how tenure, monthly charges, and total charges are distributed
across churned vs retained customers. These are likely to be strong
predictors, as customers who leave early tend to have lower tenure
and potentially higher monthly charges.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    df.groupby('Churn')[col].hist(alpha=0.6, ax=ax, bins=30)
    ax.set_title(col)
    ax.legend(['No churn', 'Churn'])
plt.tight_layout()
plt.show()

## Correlation matrix
Checking for multicollinearity between numeric features.
Highly correlated features carry redundant information — FACET will
later quantify this precisely using its redundancy decomposition.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
numeric = df[['tenure', 'MonthlyCharges', 'TotalCharges']].dropna()
plt.figure(figsize=(6, 4))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='Blues')
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

## Key business observations
We can see that:
- Short tenure customers churn at a much higher rate
- Higher monthly charges correlate with increased churn
- TotalCharges is highly correlated with tenure — likely redundant
  as a standalone feature (FACET will confirm this in Sprint 2)